In [ ]:
# from ete3 import NCBITaxa
# ncbi = NCBITaxa()

# taxids = [6183]
# lineages = ncbi.get_lineage(taxids[0]) # Returns list of parent TaxIDs
# names = ncbi.get_taxid_translator(lineages) # Translates IDs to names
# full_string = "; ".join([names[taxid] for taxid in lineages])
# print(full_string)

# lineages

In [1]:
%load_ext autoreload

In [2]:
%autoreload 3

In [3]:
import pandas as pd
from Bio import SeqIO

In [4]:
from roseate import io

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [22]:
# %% [1] Settings: edit the paths
from pathlib import Path
import subprocess
import pandas as pd
from Bio import SeqIO
from roseate import extract

WORK = Path('./../data/domain_screen').resolve()         # outputs of this notebook
HMM = Path('./../data/PF00406.hmm').resolve()
BACTERIAL_ADKS = Path('./../data/20240122_all_adks_filtered.faa').resolve()                       # GTDB bacterial ADKs
ARCHAEAL_PROTEOMES = Path('/data/mhoffert/genomes/GTDB_r214.1/protein_faa_reps/archaea').resolve()
OUTGROUP = Path('./../data/high_qual_alignments/protein-matching-TIGR01351.fasta').resolve()     # headers with taxID:NNNN
QUERY = Path('./../runs/Antarctic/1_extract/proteins.faa').resolve()        # the ADKs to screen, e.g. an extract stage output
MIN_IDENTITY = 40                                        # % identity of the best hit, below which a query is flagged
THREADS = 4
WORK.mkdir(parents=True, exist_ok=True)

In [12]:
# use ROSEATE functions to get ADKs from archaea
# %% [2] Archaeal ADKs: the same search ROSEATE uses for genomes, on GTDB's archaeal proteomes
archaea_dir = WORK / 'archaea'
proteomes = extract.find_genomes([ARCHAEAL_PROTEOMES])
extract.search_proteins(proteomes, hmm_model=HMM, out_dir=archaea_dir, jobs=THREADS)
extract.process_hmms(loc=archaea_dir, protein_name='adk', n_proteins=1, min_score=100, overwrite=True)
archaeal_adks = extract.concat_adks(loc=archaea_dir, out=WORK / 'archaeal_adks.faa', protein_name='adk')
print(sum(1 for _ in SeqIO.parse(archaeal_adks, 'fasta')), 'archaeal ADKs from', len(proteomes), 'proteomes')

Found 4416 tables to parse
Table contained no records:
/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/domain_screen/archaea/GB_GCA_000405225.1_protein_hmmsearch.tbl
Table contained no records:
/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/domain_screen/archaea/GB_GCA_000494145.1_protein_hmmsearch.tbl
Table contained no records:
/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/domain_screen/archaea/GB_GCA_000494185.1_protein_hmmsearch.tbl
Table contained no records:
/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/domain_screen/archaea/GB_GCA_000806015.1_protein_hmmsearch.tbl
Table contained no records:
/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/domain_screen/archaea/GB_GCA_001273335.1_protein_hmmsearch.tbl
Table contained no records:
/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/domain_screen/archaea/GB_GCA_001563965.1_protein_hmmsearch.tbl
Table contained no records:
/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/domain_screen/archaea/GB_GCA_001742785.1_p

In [13]:
# %% [3] Domain of each outgroup sequence, from the taxID in its header (NCBI taxonomy via ete3)
from ete3 import NCBITaxa
ncbi = NCBITaxa()                     # first use downloads the NCBI taxonomy (about 500 MB, in ~/.etetoolkit)
DOMAINS = {2: 'bac', 2157: 'arc', 2759: 'euk'}

def domain_of(taxid):
    try:
        return next((DOMAINS[t] for t in ncbi.get_lineage(taxid) if t in DOMAINS), None)
    except ValueError:                # taxID missing from the local taxonomy
        return None

outgroup = []
for r in SeqIO.parse(OUTGROUP, 'fasta'):
    taxid = int(r.description.split('taxID:')[1].split('|')[0].split()[0])
    outgroup.append((domain_of(taxid), r.id.split('|')[0], str(r.seq)))
print(pd.Series([d for d, _, _ in outgroup]).value_counts(dropna=False))

bac    585
euk     98
arc     18
Name: count, dtype: int64


In [14]:
# %% [4] Reference: every sequence named <domain>|<id>, with plain IDs for MMseqs2
reference = WORK / 'adk_reference.faa'
with open(reference, 'w') as f:
    for path, domain in [(BACTERIAL_ADKS, 'bac'), (archaeal_adks, 'arc')]:
        for n, r in enumerate(SeqIO.parse(path, 'fasta')):
            f.write(f'>{domain}|{domain}{n:06d}\n{str(r.seq).replace("*", "")}\n')
    for n, (domain, acc, seq) in enumerate(outgroup):
        if domain:
            f.write(f'>{domain}|{acc}\n{seq.replace("*", "")}\n')
print(pd.Series([r.id.split('|')[0] for r in SeqIO.parse(reference, 'fasta')]).value_counts())

bac    9117
arc    2194
euk      98
Name: count, dtype: int64


In [21]:
# %% [5] Best hit of each query in the reference
hits_file = WORK / 'hits.tsv'
subprocess.run(['mmseqs', 'easy-search', str(QUERY), str(reference), str(hits_file), str(WORK / 'tmp'),
                '--threads', str(THREADS), '--format-output', 'query,target,pident,bits'],
               check=True, capture_output=True)
hits = pd.read_csv(hits_file, sep='\t', names=['query', 'target', 'pident', 'bits'])
best = hits.sort_values('bits', ascending=False).drop_duplicates('query').set_index('query')
best['domain'] = best['target'].str.split('|').str[0]

In [23]:
# %% [6] Keep queries whose best hit is bacterial and close enough; report the rest
queries = [r.id for r in SeqIO.parse(QUERY, 'fasta')]
report = best.reindex(queries)
report['domain'] = report['domain'].fillna('no hit')
report['keep'] = (report['domain'] == 'bac') & (report['pident'] >= MIN_IDENTITY)
report.to_csv(WORK / 'domain_screen.tsv', sep='\t', index_label='query')
print(report.groupby('domain')['keep'].agg(['size', 'sum']).rename(columns={'size': 'queries', 'sum': 'kept'}))

kept = set(report.index[report['keep']])
SeqIO.write((r for r in SeqIO.parse(QUERY, 'fasta') if r.id in kept), WORK / 'bacterial_adks.faa', 'fasta')
print(f'{len(kept)} of {len(queries)} queries kept in', WORK / 'bacterial_adks.faa')

        queries  kept
domain               
arc          22     0
bac         961   950
euk           1     0
950 of 984 queries kept in /data/mhoffert/fiererlab/adenylate_kinase_ogt/data/domain_screen/bacterial_adks.faa
